<a href="https://colab.research.google.com/github/haribharadwaj/notebooks/blob/main/uPNC/ITD_nAFC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Psychometric functions for ITD discrimination

In this notebook we fit and plot **psychometric functions** for an **interaural time difference (ITD)** discrimination task. The ITD is the small difference in a sound's arrival time at the two ears, and it is one of the principal cues the auditory system uses to localize sounds. Here, listeners performed a **3-alternative forced-choice (3-AFC)** task in which they had to detect an ITD jump, and we measure how performance (percent correct) grows as the **ITD jump size** increases.

By fitting a psychometric function to the data we can estimate a **threshold** — the ITD jump size needed to reach a chosen level of performance — which gives a single, interpretable measure of each group's spatial sensitivity.

## What you'll do

1. **Set up the environment** (install the fitting package and download the data) — only needed on Google Colab.
2. **Load and aggregate** the behavioral data across listeners.
3. **Plot the empirical data** (percent correct vs. ITD jump size, with 95% confidence intervals).
4. **Fit a psychometric function** with the `psignifit` package and overlay it.
5. **Extract a threshold** from the fitted curve.

## Tools and references

We use **psignifit**, a package for Bayesian fitting of psychometric functions:

- Course fork used here: https://github.com/haribharadwaj/python-psignifit
- Upstream psignifit project: https://github.com/wichmann-lab/python-psignifit

Plotting uses **matplotlib** (via the `pylab` interface) and data handling uses **pandas** and **NumPy**.


## Setup for Google Colab

If you are running this notebook in **Google Colab**, run the next two cells **first**, before anything else.

1. The first cell installs the course fork of **psignifit** directly from GitHub, so you get the correct version.
2. The second cell downloads the behavioral data file into your Colab machine.

These steps only need to be run **once per Colab session**. If your session disconnects or you start fresh, run them again.


In [ ]:
# Install the course fork of psignifit (run once per Colab session)
!pip -q install git+https://github.com/haribharadwaj/python-psignifit

### Download the data

This cell downloads the results file `ITD_results.csv` from Dropbox into the current folder.

The CSV has five columns:

| Column  | Meaning                                              |
|:--------|:-----------------------------------------------------|
| `subj`  | Subject (listener) identifier                        |
| `ITD`   | ITD jump size in microseconds (µs)                   |
| `score` | Proportion correct for that subject at that ITD      |
| `ncorr` | Number of correct trials for that subject at that ITD|
| `ntot`  | Total number of trials for that subject at that ITD  |

> **Note:** Dropbox links must end in `dl=1` to download the raw file (a link ending in `dl=0` returns an HTML preview page instead of the data). Make sure the link below ends in `dl=1`, and keep it in quotes.


In [ ]:
# Download the behavioral data (run once per Colab session)
!wget -q --show-progress -O ITD_results.csv  "https://www.dropbox.com/scl/fi/q9u65a39mtpsi9qn17ir9/ITD_results.csv?rlkey=bnn26z88q954yst0kd8lw68ed&st=0n7b8hxx&dl=1"

# Quick check that the file arrived
!ls -lh ITD_results.csv
!head -n 3 ITD_results.csv

## Imports

We load the libraries used throughout the notebook:

- **`pandas`** — reading the CSV and grouping/aggregating the data.
- **`numpy`** — numerical arrays and the log transforms used in the fit.
- **`pylab`** (matplotlib) — plotting.
- **`psignifit`** — fitting the psychometric function.


In [ ]:
import pandas as pd
import pylab as pl
import numpy as np
import psignifit as ps

## Load the data

We read the CSV into a pandas DataFrame.

- On **Colab**, the download cell above placed the file in the current directory, so the path is simply `'ITD_results.csv'`.
- If you are running **locally** with the file elsewhere, set `froot` to the folder that contains it.

The `dat = dat0` line is where you can **filter to a subset of subjects** if you want to look at a particular group — otherwise it uses everyone. We also count the number of unique subjects `N` for display on the plot.


In [ ]:
# On Colab the file is in the current directory:
froot = ''
# If running locally, point froot at the folder containing the CSV, e.g.:
# froot = '/Users/hari/Dropbox/WebAppTest/CSD1237_ITD/Data/'

dat0 = pd.read_csv(froot + 'ITD_results.csv')
dat = dat0  # Filter subjects here if needed, e.g. dat = dat0[dat0['subj'].isin([...])]

N = len(pd.unique(dat['subj']))  # Count N for display
print(f'Number of subjects: {N}')
dat.head()

## Aggregate across listeners

For each ITD jump size we summarize performance across listeners:

- **`m`** — the mean percent-correct (`score`) at each ITD.
- **`e`** — the standard error of the mean, scaled by **1.96** to give an approximate **95% confidence interval**.
- **`ncorr` / `ntot`** — the total number of correct trials and total trials, **summed** across listeners at each ITD. These pooled counts are what the `psignifit` fit will use (it works with raw trial counts, not averaged proportions).

We convert everything to NumPy arrays, and `x` holds the ITD jump sizes (the values we'll plot on the horizontal axis).


In [ ]:
m = dat.groupby('ITD')['score'].mean()
e = dat.groupby('ITD')['score'].sem()
ncorr = dat.groupby('ITD')['ncorr'].sum()
ntot = dat.groupby('ITD')['ntot'].sum()

ntot = ntot.to_numpy()
ncorr = ncorr.to_numpy()
x = m.index.to_numpy()
m = m.to_numpy()
e = e.to_numpy() * 1.96  # scale SEM to ~95% CI

## Plot the empirical data

Now we plot mean percent-correct versus ITD jump size, with error bars showing the 95% confidence interval.

A few details:

- The **x-axis is logarithmic** (`xscale('log')`), because ITD jump sizes span a wide range and roughly double from one level to the next. The custom ticks (10, 40, 80, …, 320 µs) label the actual tested sizes.
- The y-axis is percent correct.

This cell sets up the axes and draws the data points; the **fitted curve** will be added on top of these same axes in a later cell.


In [ ]:
pl.errorbar(x, m, e, fmt='o')
pl.xscale('log')
pl.xticks(ticks=[10, 20, 40, 80, 160, 320],
          labels=['10', '20', '40', '80', '160', '320'],
          fontsize=14)
pl.yticks(fontsize=14)
pl.xlabel('ITD Jump Size (μs)', fontsize=14)
pl.ylabel('% Correct (with 95% CI)', fontsize=14)
pl.grid('on')

## Fit the psychometric function

We now fit a psychometric function to the pooled trial counts using **psignifit**.

The input `psdat` is an array with one row per ITD level and three columns:

1. the stimulus level on a **log2 scale** (`log2(x)`) — fitting in log space matches the doubling spacing of the ITDs,
2. the number of correct trials (`ncorr`),
3. the total number of trials (`ntot`).

The settings are passed to `psignifit` as **keyword arguments**:

- **`sigmoid='norm'`** — use a cumulative Gaussian as the sigmoid shape.
- **`experiment_type='3AFC'`** — a 3-alternative forced-choice task, so chance performance is **1/3 (≈33%)**; psignifit accounts for this lower asymptote automatically.
- **`thresh_PC=0.5`** — defines the threshold as the stimulus level yielding 50% correct (you can change this level to define the threshold differently).

`ps.psignifit(...)` returns a **result object** that holds the fitted parameters (`result.parameter_estimate`) and the fitting configuration (`result.configuration`, including the sigmoid we'll use to draw the curve). We can also read the threshold off it directly with `result.threshold(...)` in the next cell.

In [ ]:
psdat = np.c_[np.log2(x), ncorr, ntot]

threshPC = 0.5

# Pass settings as keyword arguments
result = ps.psignifit(
    psdat,
    sigmoid='norm',          # cumulative Gauss (was options['sigmoidName'])
    experiment_type='3AFC',  # 3-AFC (was options['expType'])
    thresh_PC=threshPC,      # threshold percent correct (was options['threshPC'])
)


## Overlay the fitted curve and extract the threshold

Finally we draw the fitted psychometric function on top of the data and read off the threshold.

- We evaluate the fitted sigmoid on a fine grid of log2 stimulus levels (`xcurve`), convert the fit's output to percent correct, and plot it (converting the x-axis back to linear µs with `2 ** xcurve`).
- **`ps.getThreshold`** returns the threshold (and its confidence interval) on the log2 scale; we convert back to µs with `2 ** ...`. This `thresh` is the ITD jump size corresponding to the `threshPC` (40%) performance level.
- We annotate the plot with the number of subjects `N`, set sensible axis limits, and add a dashed line at **33%** marking chance performance for the 3-AFC task.

The commented-out lines draw optional threshold markers (a horizontal/vertical guide to the threshold and an error bar on it) — uncomment them if you'd like those annotations.


In [ ]:
import warnings

# Evaluate the fitted curve on a fine grid (log2 stimulus levels)
xcurve = np.arange(np.log2(x[0]), np.log2(x[-1]), 0.01)

# Fit sigmoid curve (scaled by gamma and lambda)
fit = result.parameter_estimate
sigmoid = result.configuration.make_sigmoid()
fitValues = (sigmoid(xcurve, fit['threshold'], fit['width'])
             * (1 - fit['gamma'] - fit['lambda']) + fit['gamma'])

# Redraw data + overlay fit
pl.errorbar(x, m, e, fmt='o')
pl.plot(2. ** xcurve, fitValues * 100., color=[0.8, 0.8, 0.8], linewidth=2)

# Threshold and CI at threshPC, on the scaled curve. The CI from this method is an
# upper bound, so we suppress the accompanying warning.
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    thresh_log2, CI_log2 = result.threshold(threshPC)
thresh = 2. ** thresh_log2
CI = 2. ** np.asarray(CI_log2['0.95']).ravel()   # 95% upper-bound CI, in microseconds

print(f'Threshold at {int(threshPC*100)}% correct: {thresh:.1f} us '
      f'(95% CI: {CI[0]:.1f}-{CI[1]:.1f} us)')

# --- Guide arrows for "reading off" the threshold ---
ythr = threshPC * 100.        # y-level of the threshold, in percent
xleft = x.min() / 2.          # left edge (where the horizontal arrow starts)
ybot = 25.                    # bottom of plot
yci = ybot + 1.5              # place the CI bar just above the bottom axis

# Horizontal arrow: from the y-axis across to the fitted curve at threshPC
pl.annotate('', xy=(thresh, ythr), xytext=(xleft, ythr),
            arrowprops=dict(arrowstyle='->', color='k', lw=1.5))
# Vertical arrow: from the curve down to the CI bar near the x-axis
pl.annotate('', xy=(thresh, yci), xytext=(thresh, ythr),
            arrowprops=dict(arrowstyle='->', color='k', lw=1.5))

# Confidence interval around the threshold (horizontal bar near the x-axis)
pl.errorbar(thresh, yci, xerr=[[thresh - CI[0]], [CI[1] - thresh]],
            fmt='o', color='k', elinewidth=2.5, capsize=6.)

# Axis cosmetics
pl.xscale('log')
pl.xticks(ticks=[10, 20, 40, 80, 160, 320],
          labels=['10', '20', '40', '80', '160', '320'], fontsize=14)
pl.yticks(fontsize=14)
pl.xlabel('ITD Jump Size (μs)', fontsize=14)
pl.ylabel('% Correct (with 95% CI)', fontsize=14)
pl.grid('on')
pl.ylim([25., 100.])
pl.xlim([x.min()/2., x.max() * 2.])
pl.text(x.min(), 90., f'N = {N}', fontsize=14)
pl.hlines(33, x.min()/2., x.max()*2., 'k', linestyles='--')
pl.show()